<a href="https://colab.research.google.com/github/tiendungnguyenw0907-boop/DocumentIntelligence/blob/main/_notebooks/2026-06-25-ReadPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Layer 1. Physical Understanding**

## 1. Ingestion

> *Tiếp nhận tài liệu từ các nguồn khác nhau và đánh giá chất lượng đầu vào nhằm bảo đảm tài liệu đủ điều kiện để đưa vào quy trình xử lý tự động. Đây là bước kiểm soát chất lượng đầu vào của toàn bộ hệ thống*



### 1.1 Tiếp nhận tài liệu

### 1.2 Kiểm tra chất lượng

# 2. Physical Analysis

> *Phân tích các đặc tính vật lý và kỹ thuật của tài liệu nhằm hiểu tài liệu được tạo ra như thế nào, gồm những thành phần gì và cần áp dụng chiến lược xử lý nào. Kết quả của bước này giúp hệ thống xác định phương pháp OCR, nhận dạng bố cục và trích xuất thông tin phù hợp*



## 2.1. PDF Parsing & Format Analysis
> **Mục tiêu**
> *   *Hiểu bản chất vật lý của tài liệu trước khi xử lý*
> *   *Xác định loại PDF và chiến lược xử lý phù hợp*
> *   *Phân tích cấu trúc kỹ thuật của tài liệu*

> **Bài toán cần giải quyết**
>   Tài liệu không chỉ có văn bản
>   Một trang có thể chứa (*Tiêu đề, Đoạn văn, Bảng biểu, Hình ảnh, Header, Footer, Chú thích*)

> **Do đó**
>*   *OCR toàn trang dễ làm mất ngữ cảnh*
>*   *Không xác định được ranh giới các đối tượng*
>*   *Khó phục hồi cấu trúc tài liệu sau OCR*


### 2.1.1. Phân tích cấu trúc PDF

#### 2.1.1.1 Installation

In [1]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 66.6 MB/s eta 0:00:00


#### 2.1.1.2. UPLOAD PDF




In [2]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("PDF:", pdf_file)

Saving SachBCTHKQKT2021.pdf to SachBCTHKQKT2021.pdf
PDF: SachBCTHKQKT2021.pdf


#### 2.1.1.3. PDF Structure Analyzer

In [3]:
import fitz
import json
import os

from tqdm import tqdm
from collections import Counter


class PDFStructureAnalyzer:

    def __init__(self, pdf_path):

        self.pdf_path = pdf_path

        self.doc = fitz.open(pdf_path)

    def analyze(self):

        file_size_mb = round(
            os.path.getsize(self.pdf_path) / 1024 / 1024,
            2
        )

        result = {

            "document_id": "DOC001",

            "file_info": {},

            "document_structure": {},

            "object_statistics": {},

            "font_statistics": [],

            "page_structure": [],

            "page_summary": {},

            "technical_flags": {}

        }

        result["file_info"] = {

            "file_name": os.path.basename(self.pdf_path),

            "file_size_mb": file_size_mb,

            "pdf_version": "Unknown",

            "encrypted": self.doc.needs_pass,

            "page_count": len(self.doc)

        }

        total_text_objects = 0

        total_image_objects = 0

        total_annotation_objects = 0

        total_vector_objects = 0

        font_counter = Counter()

        pages_with_text = 0

        pages_without_text = 0

        pages_with_images = 0

        pages_with_annotations = 0

        rotated_pages = 0

        has_bookmark = False

        has_hyperlink = False

        has_text_layer = False

        has_image_layer = False

        page_structure = []

        print("Analyzing pages...")

        for page_num in tqdm(range(len(self.doc))):

            page = self.doc[page_num]

            try:

                text = page.get_text()

            except:

                text = ""

            try:

                text_dict = page.get_text("dict")

                blocks = text_dict.get("blocks", [])

                text_object_count = len(blocks)

            except:

                text_object_count = 0

            try:

                image_count = len(page.get_images(full=True))

            except:

                image_count = 0

            try:

                fonts = page.get_fonts()

            except:

                fonts = []

            font_names = []

            for f in fonts:

                if len(f) > 3:

                    font_name = str(f[3])

                    font_names.append(font_name)

                    font_counter[font_name] += 1

            annotation_count = 0

            try:

                annots = page.annots()

                if annots:

                    annotation_count = len(list(annots))

            except:

                pass

            try:

                links = page.get_links()

                if len(links) > 0:

                    has_hyperlink = True

            except:

                pass

            page_info = {

                "page_number": page_num + 1,

                "has_text_layer": len(text.strip()) > 0,

                "text_object_count": text_object_count,

                "image_object_count": image_count,

                "font_count": len(set(font_names)),

                "annotation_count": annotation_count,

                "page_rotation": page.rotation

            }

            page_structure.append(page_info)

            total_text_objects += text_object_count

            total_image_objects += image_count

            total_annotation_objects += annotation_count

            if len(text.strip()) > 0:

                pages_with_text += 1

                has_text_layer = True

            else:

                pages_without_text += 1

            if image_count > 0:

                pages_with_images += 1

                has_image_layer = True

            if annotation_count > 0:

                pages_with_annotations += 1

            if page.rotation != 0:

                rotated_pages += 1

        try:

            toc = self.doc.get_toc()

            has_bookmark = len(toc) > 0

        except:

            pass

        result["document_structure"] = {

            "has_text_layer": has_text_layer,

            "has_image_layer": has_image_layer,

            "has_annotation": pages_with_annotations > 0,

            "has_form_field": False,

            "has_embedded_font": len(font_counter) > 0,

            "has_vector_graphic": False,

            "has_bookmark": has_bookmark,

            "has_hyperlink": has_hyperlink,

            "has_watermark": False

        }

        result["object_statistics"] = {

            "total_text_objects": total_text_objects,

            "total_image_objects": total_image_objects,

            "total_font_objects": len(font_counter),

            "total_annotation_objects": total_annotation_objects,

            "total_vector_objects": total_vector_objects

        }

        font_statistics = []

        for font_name, count in font_counter.most_common():

            font_statistics.append({

                "font_name": font_name,

                "usage_count": count

            })

        result["font_statistics"] = font_statistics

        result["page_structure"] = page_structure

        result["page_summary"] = {

            "pages_with_text_layer": pages_with_text,

            "pages_without_text_layer": pages_without_text,

            "pages_with_images": pages_with_images,

            "pages_with_annotations": pages_with_annotations,

            "rotated_pages": rotated_pages

        }

        result["technical_flags"] = {

            "contains_scanned_pages":
                pages_without_text > 0,

            "contains_native_pages":
                pages_with_text > 0,

            "contains_mixed_pages":
                pages_with_text > 0 and pages_without_text > 0,

            "contains_large_images":
                total_image_objects > 100,

            "contains_rotated_pages":
                rotated_pages > 0

        }

        return result

#### 2.1.1.4. Chạy phân tích

In [4]:
analyzer = PDFStructureAnalyzer(pdf_file)

result = analyzer.analyze()

print("Done")

Analyzing pages...


100%|██████████| 284/284 [00:02<00:00, 111.65it/s]

Done


#### 2.1.1.5. Xuất JSON

In [6]:
from pathlib import Path
OUTPUT_DIR = Path("output")
RENDER_DIR = OUTPUT_DIR / "1.Physical_Analysis" / "1.PDF_Parsing_Format_Analysis"
RENDER_DIR.mkdir(parents=True, exist_ok=True)
output_pdf_structure_profile_file = RENDER_DIR /"1.pdf_structure_profile.json"

with open(
    output_pdf_structure_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", output_pdf_structure_profile_file)

Saved: output/1.Physical_Analysis/1.PDF_Parsing_Format_Analysis/1.pdf_structure_profile.json


#### 2.1.1.6. Download JSON

In [ ]:
from google.colab import files

files.download(output_pdf_structure_profile_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 2.1.2. Xác định loại PDF

#### 2.1.2.1 Load PDF Structure Profile

In [7]:
import json

with open(output_pdf_structure_profile_file, "r", encoding="utf-8") as f:
    structure_profile = json.load(f)

print("Loaded")

Loaded


#### 2.1.2.2 PDF Type Analyzer

In [8]:
class PDFTypeAnalyzer:

    def __init__(self, structure_profile):

        self.structure_profile = structure_profile

    def analyze(self):

        page_structure = self.structure_profile["page_structure"]

        total_pages = len(page_structure)

        native_pages = []
        scan_pages = []
        mixed_pages = []

        for page in page_structure:

            has_text = page["has_text_layer"]

            image_count = page["image_object_count"]

            if has_text and image_count == 0:

                native_pages.append(
                    page["page_number"]
                )

            elif (not has_text) and image_count > 0:

                scan_pages.append(
                    page["page_number"]
                )

            else:

                mixed_pages.append(
                    page["page_number"]
                )

        native_count = len(native_pages)

        scan_count = len(scan_pages)

        mixed_count = len(mixed_pages)

        native_ratio = round(
            native_count / total_pages * 100,
            2
        )

        scan_ratio = round(
            scan_count / total_pages * 100,
            2
        )

        mixed_ratio = round(
            mixed_count / total_pages * 100,
            2
        )

        # ==========================
        # Determine PDF Type
        # ==========================

        if scan_count == 0 and mixed_count == 0:

            pdf_type = "NATIVE"

        elif native_count == 0 and mixed_count == 0:

            pdf_type = "SCAN"

        else:

            pdf_type = "HYBRID"

        # ==========================
        # OCR Strategy
        # ==========================

        if pdf_type == "NATIVE":

            ocr_required = False

            ocr_mode = "NONE"

        elif pdf_type == "SCAN":

            ocr_required = True

            ocr_mode = "FULL_DOCUMENT"

        else:

            ocr_required = True

            ocr_mode = "PAGE_BASED"

        # ==========================
        # Layout Strategy
        # ==========================

        if pdf_type == "NATIVE":

            layout_strategy = "TEXT_FIRST"

        elif pdf_type == "SCAN":

            layout_strategy = "OCR_FIRST"

        else:

            layout_strategy = "HYBRID"

        # ==========================
        # Processing Strategy
        # ==========================

        if pdf_type == "NATIVE":

            processing_strategy = "DIRECT_LAYOUT"

        elif pdf_type == "SCAN":

            processing_strategy = "OCR_LAYOUT"

        else:

            processing_strategy = "MIXED"

        result = {

            "pdf_type": pdf_type,

            "page_count": total_pages,

            "native_pages": native_count,

            "scan_pages": scan_count,

            "mixed_pages": mixed_count,

            "native_ratio": native_ratio,

            "scan_ratio": scan_ratio,

            "mixed_ratio": mixed_ratio,

            "ocr_required": ocr_required,

            "ocr_mode": ocr_mode,

            "layout_strategy": layout_strategy,

            "processing_strategy": processing_strategy,

            "page_groups": {

                "native_pages": native_pages,

                "scan_pages": scan_pages,

                "mixed_pages": mixed_pages

            }

        }

        return result

#### 2.1.2.3 Run

In [9]:
analyzer = PDFTypeAnalyzer(
    structure_profile
)

pdf_type_profile = analyzer.analyze()

print(
    json.dumps(
        pdf_type_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "pdf_type": "NATIVE",
  "page_count": 284,
  "native_pages": 284,
  "scan_pages": 0,
  "mixed_pages": 0,
  "native_ratio": 100.0,
  "scan_ratio": 0.0,
  "mixed_ratio": 0.0,
  "ocr_required": false,
  "ocr_mode": "NONE",
  "layout_strategy": "TEXT_FIRST",
  "processing_strategy": "DIRECT_LAYOUT",
  "page_groups": {
    "native_pages": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      6

#### 2.1.2.4 Save JSON

In [10]:
output_pdf_type_profile_file = RENDER_DIR /"2.pdf_type_profile.json"

with open(output_pdf_type_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pdf_type_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved")

Saved


### 2.1.3. Phân tích nội dung tài liệu

#### 2.1.3.1. Code

In [11]:
import fitz
import json

class DocumentContentAnalyzer:

    def __init__(
        self,
        pdf_file,
        structure_profile,
        pdf_type_profile
    ):

        self.pdf_file = pdf_file

        self.structure_profile = structure_profile

        self.pdf_type_profile = pdf_type_profile

        self.doc = fitz.open(pdf_file)

    def analyze(self):

        contains_text = False
        contains_image = False
        contains_table = False
        contains_form = False
        contains_hyperlink = False
        contains_bookmark = False
        contains_signature = False
        contains_stamp = False

        pages_with_text = 0
        pages_with_images = 0
        estimated_table_pages = 0
        pages_with_links = 0
        pages_with_forms = 0

        # =====================
        # Bookmark
        # =====================

        try:

            toc = self.doc.get_toc()

            if len(toc) > 0:

                contains_bookmark = True

        except Exception:

            pass

        # =====================
        # Page Analysis
        # =====================

        for page in self.doc:

            # -------------------
            # Text
            # -------------------

            text = page.get_text()

            if text.strip():

                contains_text = True

                pages_with_text += 1

            # -------------------
            # Images
            # -------------------

            images = page.get_images(
                full=True
            )

            if len(images) > 0:

                contains_image = True

                pages_with_images += 1

            # -------------------
            # Hyperlinks
            # -------------------

            links = page.get_links()

            if len(links) > 0:

                contains_hyperlink = True

                pages_with_links += 1

            # -------------------
            # Forms
            # -------------------

            try:

                widgets = page.widgets()

                if widgets:

                    contains_form = True

                    pages_with_forms += 1

            except Exception:

                pass

            # -------------------
            # Table Estimation
            #
            # Rule-based
            # -------------------

            text_dict = page.get_text(
                "dict"
            )

            block_count = len(
                text_dict.get(
                    "blocks",
                    []
                )
            )

            if block_count >= 20:

                estimated_table_pages += 1

                contains_table = True

        result = {

            "document_content_profile": {

                "contains_text":
                    contains_text,

                "contains_image":
                    contains_image,

                "contains_table":
                    contains_table,

                "contains_form":
                    contains_form,

                "contains_hyperlink":
                    contains_hyperlink,

                "contains_bookmark":
                    contains_bookmark,

                "contains_signature":
                    contains_signature,

                "contains_stamp":
                    contains_stamp
            },

            "content_statistics": {

                "pages_with_text":
                    pages_with_text,

                "pages_with_images":
                    pages_with_images,

                "estimated_table_pages":
                    estimated_table_pages,

                "pages_with_links":
                    pages_with_links,

                "pages_with_forms":
                    pages_with_forms
            }
        }

        return result

#### 2.1.3.2. Run

In [12]:
with open(
    output_pdf_structure_profile_file,
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    output_pdf_type_profile_file,
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

analyzer = DocumentContentAnalyzer(
    pdf_file,
    structure_profile,
    pdf_type_profile
)

content_profile = analyzer.analyze()

print(
    json.dumps(
        content_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_content_profile": {
    "contains_text": true,
    "contains_image": false,
    "contains_table": true,
    "contains_form": true,
    "contains_hyperlink": false,
    "contains_bookmark": false,
    "contains_signature": false,
    "contains_stamp": false
  },
  "content_statistics": {
    "pages_with_text": 284,
    "pages_with_images": 0,
    "estimated_table_pages": 184,
    "pages_with_links": 0,
    "pages_with_forms": 284
  }
}


#### 2.1.3.3. Lưu file

In [13]:
output_document_content_profile_file = RENDER_DIR /"3.document_content_profile.json"
with open(
    output_document_content_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        content_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

### 2.1.4. Phân tích phân bố dữ liệu

#### 2.1.4.1 Code

In [14]:
import json

class ContentDistributionAnalyzer:

    def __init__(
        self,
        structure_profile,
        pdf_type_profile,
        content_profile
    ):

        self.structure_profile = structure_profile

        self.pdf_type_profile = pdf_type_profile

        self.content_profile = content_profile

    def calculate_page_complexity(
        self,
        text_objects,
        image_objects,
        font_count,
        annotation_count
    ):

        score = (

            text_objects * 0.25

            +

            image_objects * 10

            +

            font_count * 2

            +

            annotation_count * 5

        )

        return min(
            round(score),
            100
        )

    def classify_complexity(
        self,
        score
    ):

        if score >= 80:

            return "HIGH"

        elif score >= 50:

            return "MEDIUM"

        else:

            return "LOW"

    def analyze(self):

        pages = self.structure_profile[
            "page_structure"
        ]

        page_profiles = []

        total_text_density = 0

        total_image_density = 0

        total_content_density = 0

        total_complexity = 0

        for page in pages:

            page_number = page[
                "page_number"
            ]

            text_objects = page.get(
                "text_object_count",
                0
            )

            image_objects = page.get(
                "image_object_count",
                0
            )

            font_count = page.get(
                "font_count",
                0
            )

            annotation_count = page.get(
                "annotation_count",
                0
            )

            total_objects = max(
                text_objects + image_objects,
                1
            )

            text_density = round(
                text_objects
                / total_objects
                * 100,
                2
            )

            image_density = round(
                image_objects
                / total_objects
                * 100,
                2
            )

            content_density = min(
                round(
                    (
                        text_objects * 0.2
                        +
                        image_objects * 8
                    )
                ),
                100
            )

            complexity_score = (
                self.calculate_page_complexity(
                    text_objects,
                    image_objects,
                    font_count,
                    annotation_count
                )
            )

            page_complexity = (
                self.classify_complexity(
                    complexity_score
                )
            )

            page_profiles.append({

                "page_number":
                    page_number,

                "text_density":
                    text_density,

                "image_density":
                    image_density,

                "content_density":
                    content_density,

                "complexity_score":
                    complexity_score,

                "page_complexity":
                    page_complexity
            })

            total_text_density += text_density

            total_image_density += image_density

            total_content_density += content_density

            total_complexity += complexity_score

        page_count = len(
            page_profiles
        )

        avg_text_density = round(
            total_text_density
            / page_count,
            2
        )

        avg_image_density = round(
            total_image_density
            / page_count,
            2
        )

        avg_content_density = round(
            total_content_density
            / page_count,
            2
        )

        avg_complexity = round(
            total_complexity
            / page_count,
            2
        )

        if avg_complexity >= 80:

            document_complexity = "HIGH"

        elif avg_complexity >= 50:

            document_complexity = "MEDIUM"

        else:

            document_complexity = "LOW"

        result = {

            "document_distribution_profile": {

                "average_text_density":
                    avg_text_density,

                "average_image_density":
                    avg_image_density,

                "average_content_density":
                    avg_content_density,

                "average_complexity_score":
                    avg_complexity,

                "document_complexity":
                    document_complexity
            },

            "page_distribution_profiles":
                page_profiles
        }

        return result

#### 2.1.4.2 Run

In [15]:
with open(
    output_pdf_structure_profile_file,
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    output_pdf_type_profile_file,
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

with open(
    output_document_content_profile_file,
    "r",
    encoding="utf-8"
) as f:

    content_profile = json.load(f)

analyzer = ContentDistributionAnalyzer(
    structure_profile,
    pdf_type_profile,
    content_profile
)

distribution_profile = analyzer.analyze()

print(
    json.dumps(
        distribution_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_distribution_profile": {
    "average_text_density": 100.0,
    "average_image_density": 0.0,
    "average_content_density": 4.6,
    "average_complexity_score": 11.05,
    "document_complexity": "LOW"
  },
  "page_distribution_profiles": [
    {
      "page_number": 1,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 3,
      "complexity_score": 9,
      "page_complexity": "LOW"
    },
    {
      "page_number": 2,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 0,
      "complexity_score": 2,
      "page_complexity": "LOW"
    },
    {
      "page_number": 3,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 6,
      "complexity_score": 12,
      "page_complexity": "LOW"
    },
    {
      "page_number": 4,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 6,
      "complexity_score": 11,
      "page_complexity": "LOW"
    },
    {
      "pa

#### 2.1.4.3. Lưu kết quả

In [16]:
output_content_distribution_profile_file = RENDER_DIR /"4.content_distribution_profile.json"
with open(
    output_content_distribution_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        distribution_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved")

Saved


### 2.1.5. Sinh hồ sơ cấu trúc tài liệu

#### 2.1.5.1. Code

In [17]:
import json

class DocumentProfileBuilder:

    def __init__(
        self,
        structure_profile,
        pdf_type_profile,
        content_profile,
        distribution_profile
    ):

        self.structure_profile = structure_profile
        self.pdf_type_profile = pdf_type_profile
        self.content_profile = content_profile
        self.distribution_profile = distribution_profile

    def build(self):

        file_info = self.structure_profile.get(
            "file_info",
            {}
        )

        page_summary = self.structure_profile.get(
            "page_summary",
            {}
        )

        object_statistics = self.structure_profile.get(
            "object_statistics",
            {}
        )

        page_structure = self.structure_profile.get(
            "page_structure",
            []
        )

        document_content = self.content_profile.get(
            "document_content_profile",
            {}
        )

        distribution = self.distribution_profile.get(
            "document_distribution_profile",
            {}
        )

        document_profile = {

            "document_id":

                self.structure_profile.get(
                    "document_id",
                    "DOC001"
                ),

            # ==========================
            # File Information
            # ==========================

            "file_info": {

                "file_name":
                    file_info.get(
                        "file_name"
                    ),

                "page_count":
                    file_info.get(
                        "page_count"
                    ),

                "pdf_version":
                    file_info.get(
                        "pdf_version"
                    ),

                "pdf_type":
                    self.pdf_type_profile.get(
                        "pdf_type"
                    )
            },

            # ==========================
            # Processing Profile
            # ==========================

            "processing_profile": {

                "ocr_required":
                    self.pdf_type_profile.get(
                        "ocr_required"
                    ),

                "ocr_mode":
                    self.pdf_type_profile.get(
                        "ocr_mode"
                    ),

                "layout_strategy":
                    self.pdf_type_profile.get(
                        "layout_strategy"
                    ),

                "processing_strategy":
                    self.pdf_type_profile.get(
                        "processing_strategy"
                    )
            },

            # ==========================
            # Physical Profile
            # ==========================

            "physical_profile": {

                "contains_text":
                    document_content.get(
                        "contains_text"
                    ),

                "contains_image":
                    document_content.get(
                        "contains_image"
                    ),

                "contains_table":
                    document_content.get(
                        "contains_table"
                    ),

                "contains_form":
                    document_content.get(
                        "contains_form"
                    ),

                "contains_hyperlink":
                    document_content.get(
                        "contains_hyperlink"
                    ),

                "contains_bookmark":
                    document_content.get(
                        "contains_bookmark"
                    ),

                "contains_signature":
                    document_content.get(
                        "contains_signature"
                    ),

                "contains_stamp":
                    document_content.get(
                        "contains_stamp"
                    ),

                "pages_with_text":
                    page_summary.get(
                        "pages_with_text_layer"
                    ),

                "pages_with_images":
                    page_summary.get(
                        "pages_with_images"
                    ),

                "pages_with_annotations":
                    page_summary.get(
                        "pages_with_annotations"
                    ),

                "rotated_pages":
                    page_summary.get(
                        "rotated_pages"
                    )
            },

            # ==========================
            # Object Statistics
            # ==========================

            "object_statistics": {

                "total_text_objects":
                    object_statistics.get(
                        "total_text_objects"
                    ),

                "total_image_objects":
                    object_statistics.get(
                        "total_image_objects"
                    ),

                "total_font_objects":
                    object_statistics.get(
                        "total_font_objects"
                    ),

                "total_annotation_objects":
                    object_statistics.get(
                        "total_annotation_objects"
                    )
            },

            # ==========================
            # Distribution Profile
            # ==========================

            "distribution_profile": {

                "average_text_density":
                    distribution.get(
                        "average_text_density"
                    ),

                "average_image_density":
                    distribution.get(
                        "average_image_density"
                    ),

                "average_content_density":
                    distribution.get(
                        "average_content_density"
                    ),

                "average_complexity_score":
                    distribution.get(
                        "average_complexity_score"
                    ),

                "document_complexity":
                    distribution.get(
                        "document_complexity"
                    )
            },

            # ==========================
            # Page Structure
            # ==========================

            "page_structure":

                page_structure
        }

        return document_profile

#### 2.1.5.2. Run

In [18]:
with open(
    output_pdf_structure_profile_file,
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    output_pdf_type_profile_file,
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

with open(
    output_document_content_profile_file,
    "r",
    encoding="utf-8"
) as f:

    content_profile = json.load(f)

with open(
    output_content_distribution_profile_file,
    "r",
    encoding="utf-8"
) as f:

    distribution_profile = json.load(f)

builder = DocumentProfileBuilder(
    structure_profile,
    pdf_type_profile,
    content_profile,
    distribution_profile
)

document_profile = builder.build()

#### 2.1.5.3. Lưu kết quả

In [19]:
output_document_profile_file = RENDER_DIR /"5.document_profile.json"
with open(
    output_document_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        document_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

## 2.2. Layout Pre-Detection
> **Mục tiêu**
> *   *Hiểu bố cục vật lý của từng trang tài liệu*
> *   *Xác định vị trí và ranh giới của các vùng nội dung*
> *   *Phân loại các đối tượng hiển thị trước khi OCR*
> *   *Xây dựng Layout Map và Layout Tree làm nền tảng cho các bước xử lý tiếp theo*

> **Bài toán cần giải quyết**: Một trang tài liệu có nhiều loại đối tượng khác nhau (*Heading, Paragraph, Table, Image, Header, Footer, Footnote, Signature, Stamp*)

> **Do đó**
> *   *OCR toàn trang sẽ làm mất cấu trúc*
> *   *Không xác định được ranh giới giữa các vùng*
> *   *Không thể xác định thứ tự đọc chính xác*
> *   *Khó nhận diện bảng nhiều trang*
  

### 2.2.1. Page Rendering

> Kết xuất các trạng PDF theo document_profile.json từ bước 1



#### 2.2.1.1. Config

In [20]:
import json
import time
import hashlib
from pathlib import Path
import fitz
from PIL import Image


In [21]:
RENDER_DIR = OUTPUT_DIR / "1.Physical_Analysis" / "2.Layout_Pre_Detection"
RENDER_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_DIR = RENDER_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

PDF_FILE = pdf_file
DOCUMENT_PROFILE = output_document_profile_file
PAGE_RENDER_PROFILE = RENDER_DIR / "1.page_render_profile.json"

#### 2.2.1.2. Process

In [22]:
class PageRenderer:

    def __init__(self, pdf_file, profile_file):
        self.pdf_file = Path(pdf_file)
        self.profile_file = Path(profile_file)

        if not self.pdf_file.exists():
            raise FileNotFoundError(self.pdf_file)

        if not self.profile_file.exists():
            raise FileNotFoundError(self.profile_file)

        with open(self.profile_file, "r", encoding="utf-8") as f:
            self.profile = json.load(f)

        self.doc = fitz.open(self.pdf_file)

    def build_strategy(self):
        processing = self.profile.get("processing_profile", {})
        file_info = self.profile.get("file_info", {})

        pdf_type = file_info.get("pdf_type", "NATIVE")

        if pdf_type == "SCAN":
            dpi = 300
            color = "L"
        elif pdf_type == "HYBRID":
            dpi = 250
            color = "RGB"
        else:
            dpi = 200
            color = "RGB"

        return {
            "pdf_type": pdf_type,
            "dpi": dpi,
            "color_mode": color,
            "ocr_mode": processing.get("ocr_mode", "NONE"),
            "processing_strategy": processing.get(
                "processing_strategy",
                "DIRECT_LAYOUT"
            )
        }

    @staticmethod
    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            while True:
                b = f.read(1024 * 1024)
                if not b:
                    break
                h.update(b)
        return h.hexdigest()

    def render(self):
        strategy = self.build_strategy()

        scale = strategy["dpi"] / 72.0

        result = {
            "render_engine": "PyMuPDF",
            "render_strategy": strategy,
            "pages": []
        }

        total = len(self.doc)

        for idx in range(total):

            page = self.doc.load_page(idx)

            start = time.time()

            pix = page.get_pixmap(
                matrix=fitz.Matrix(scale, scale),
                alpha=False
            )

            image_path = IMAGE_DIR / f"page_{idx+1:05d}.png"

            pix.save(str(image_path))

            if strategy["color_mode"] == "L":
                img = Image.open(image_path).convert("L")
                img.save(image_path)

            elapsed = round((time.time() - start) * 1000, 2)

            result["pages"].append({
                "page_number": idx + 1,
                "image_file": image_path.name,
                "width": pix.width,
                "height": pix.height,
                "dpi": strategy["dpi"],
                "render_time_ms": elapsed,
                "sha256": self.sha256_file(image_path)
            })

            print(f"Rendered {idx+1}/{total}")

            del pix

        with open(PAGE_RENDER_PROFILE, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=4)

        print("Done.")
        print("Profile:", PAGE_RENDER_PROFILE)


#### 2.2.1.3. Run

In [23]:
renderer = PageRenderer(
    PDF_FILE,
    DOCUMENT_PROFILE
)
renderer.render()

Rendered 1/284
Rendered 2/284
Rendered 3/284
Rendered 4/284
Rendered 5/284
Rendered 6/284
Rendered 7/284
Rendered 8/284
Rendered 9/284
Rendered 10/284
Rendered 11/284
Rendered 12/284
Rendered 13/284
Rendered 14/284
Rendered 15/284
Rendered 16/284
Rendered 17/284
Rendered 18/284
Rendered 19/284
Rendered 20/284
Rendered 21/284
Rendered 22/284
Rendered 23/284
Rendered 24/284
Rendered 25/284
Rendered 26/284
Rendered 27/284
Rendered 28/284
Rendered 29/284
Rendered 30/284
Rendered 31/284
Rendered 32/284
Rendered 33/284
Rendered 34/284
Rendered 35/284
Rendered 36/284
Rendered 37/284
Rendered 38/284
Rendered 39/284
Rendered 40/284
Rendered 41/284
Rendered 42/284
Rendered 43/284
Rendered 44/284
Rendered 45/284
Rendered 46/284
Rendered 47/284
Rendered 48/284
Rendered 49/284
Rendered 50/284
Rendered 51/284
Rendered 52/284
Rendered 53/284
Rendered 54/284
Rendered 55/284
Rendered 56/284
Rendered 57/284
Rendered 58/284
Rendered 59/284
Rendered 60/284
Rendered 61/284
Rendered 62/284
Rendered 63/284
R

### 2.2.2. Layout Detection

#### 2.2.2.1. Installation

In [24]:
!pip uninstall -y paddlepaddle paddleocr paddlex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


#### 2.2.2.2. Import

In [39]:
import os
os.environ["FLAGS_enable_pir_api"] = "0"

from paddleocr import PPStructureV3
import cv2
import json
from pathlib import Path


import paddle
import paddleocr

print("Paddle :", paddle.__version__)
print("OCR    :", paddleocr.__version__)

Paddle : 3.3.1
OCR    : 3.7.0


#### 2.2.2.3. Khởi tạo

In [28]:
pipeline = PPStructureV3(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False
)

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocBlockLayout', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-DocBlockLayout), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-DocBlockLayout`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your s

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-DocLayout_plus-L', None, None)
Using official model (PP-DocLayout_plus-L), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-DocLayout_plus-L`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('RT-DETR-L_wireless_table_cell_det', None, None)
Using official model (RT-DETR-L_wireless_table_cell_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/RT-DETR-L_wireless_table_cell_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-FormulaNet_plus-L', None, None)
Using official model (PP-FormulaNet_plus-L), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-FormulaNet_plus-L`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

#### 2.2.2.4. Đọc ảnh

In [33]:
image_file_test = IMAGE_DIR / "page_00004.png"
image = cv2.imread(image_file_test)


#### 2.2.2.5. Chạy Layout Detection và xem kết quả

In [34]:
result = pipeline.predict(image)
for item in result:
    print(item)

layout = []

idx = 1

for obj in result:

    layout.append({

        "object_id":f"OBJ{idx:05d}",

        "type":obj["label"],

        "bbox":obj["coordinate"],

        "confidence":obj["score"]

    })

    idx += 1

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)


#### 2.2.2.6. Lưu kết quả

In [ ]:
OUTPUT_DIR = Path("output")
RENDER_DIR = OUTPUT_DIR / "2.Layout_Pre_Detection"
LAYOUT_OBJECTS_FILE = RENDER_DIR / "2.layout_objects.json"
with open(
    LAYOUT_OBJECTS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        layout,
        f,
        ensure_ascii=False,
        indent=4
    )

### 2.2.3. Region Classification

### 2.2.4. Bounding Box Generation

### 2.2.5. Layout Tree Construction

### 2.2.6. Layout Metadata Construction